# Roadmapify — Phase 2: Data Collection, Preprocessing & EDA

**Group:** F26-17 | **Track:** A — Application Development  
**Course:** Artificial Intelligence — FAST NUCES Lahore, Spring 2026  
**Instructor:** Hajra Waheed  

| ID | Name | Role |
|----|------|------|
| 23L-0624 | Burhan Shahzad | Group Lead |
| 23L-0589 | Ayesha Shahid | Member |
| 23L-0726 | Dua Khurram | Member |
| 23L-0556 | Syeda Manahil Atif | Member |

---

## Overview

This notebook documents the complete data pipeline for **Roadmapify** — an AI-powered learning roadmap generator.  
It covers:
1. Data sources and collection methodology
2. Raw data loading and inspection
3. Cleaning and preprocessing
4. Chunking and metadata labeling
5. ChromaDB embedding and storage
6. Exploratory Data Analysis (EDA) with visualizations


## 1. Setup & Imports

In [ ]:
import json
import re
import pathlib
import warnings
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")

# Plotting style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "DejaVu Sans",
})
PALETTE = ["#4F81BD", "#C0504D", "#9BBB59", "#8064A2", "#4BACC6", "#F79646", "#2C4770"]

print("✓ All imports successful")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")


## 2. Dataset Sources & Collection Method

Roadmapify's knowledge base is built from **12 distinct sources** spanning developer roadmaps,
academic syllabi, community Q&A, tutorial platforms, and occupational databases.

| # | Source | Type | Domains Covered | Method |
|---|--------|------|----------------|--------|
| 1 | roadmap.sh | Roadmap trees | All dev domains | BeautifulSoup scraping |
| 2 | freeCodeCamp | Tutorial articles | Web dev, Python, DSA | BeautifulSoup scraping |
| 3 | WikiHow | How-to guides | Cooking, crochet, IELTS, marketing | BeautifulSoup scraping |
| 4 | Instructables | Step-by-step guides | Crafts, cooking, baking | BeautifulSoup scraping |
| 5 | MIT OCW | Academic syllabi | CS, algorithms, linear algebra | BeautifulSoup scraping |
| 6 | CS50 (Harvard) | Course outlines | CS, Python, web dev, AI | BeautifulSoup scraping |
| 7 | YouTube transcripts | Video transcripts | All domains | youtube-transcript-api |
| 8 | Reddit (PRAW) | Community Q&A | All domains | PRAW (Reddit API) |
| 9 | Kaggle / Coursera CSV | Course metadata | All domains | Kaggle API + CSV |
| 10 | Tavily Search API | Live web search | All domains | Tavily API |
| 11 | O*NET Database | Occupational data | Professional domains | Bulk CSV download |
| 12 | Curated JSON files | Seed knowledge | 10 specific domains | Manual authoring |

**Total target coverage:** 15+ learning domains including web development, data science,
UI/UX design, digital marketing, IELTS preparation, cooking, baking, crochet, and language learning.


## 3. Loading Raw Data

In [ ]:
# ── Load all raw JSON files from data/raw/ ────────────────────────────────────

RAW_DIR = pathlib.Path("data/raw")
raw_files = sorted(RAW_DIR.glob("*.json"))

all_raw_docs = []
source_counts = {}

for path in raw_files:
    with open(path, encoding="utf-8") as f:
        try:
            docs = json.load(f)
            if isinstance(docs, list):
                source_counts[path.stem] = len(docs)
                all_raw_docs.extend(docs)
                print(f"  {path.name:<35} {len(docs):>5} documents")
        except json.JSONDecodeError:
            print(f"  {path.name:<35} [ERROR: invalid JSON]")

print(f"\n{'─'*50}")
print(f"  {'TOTAL':<35} {len(all_raw_docs):>5} documents")

# Also load curated JSON files
CURATED_DIR = pathlib.Path("data/curated")
curated_docs = []
if CURATED_DIR.exists():
    for path in sorted(CURATED_DIR.glob("*.json")):
        with open(path, encoding="utf-8") as f:
            try:
                data = json.load(f)
                # Curated files have their own structure — convert to standard format
                if isinstance(data, dict):
                    content = json.dumps(data, ensure_ascii=False)[:4000]
                    curated_docs.append({
                        "source":       "curated_json",
                        "domain":       path.stem,
                        "roadmap":      path.stem.replace("_", " ").title(),
                        "topic":        path.stem.replace("_", " ").title(),
                        "url":          "",
                        "content":      content,
                        "content_type": "curated_domain",
                    })
            except Exception:
                pass
    print(f"  {'curated_json (10 files)':<35} {len(curated_docs):>5} documents")

all_raw_docs.extend(curated_docs)
print(f"\n  {'GRAND TOTAL':<35} {len(all_raw_docs):>5} documents")


### 3.1 Raw Data Overview

In [ ]:
# Convert to DataFrame for analysis
df_raw = pd.DataFrame(all_raw_docs)

print("Shape:", df_raw.shape)
print("\nColumns:", list(df_raw.columns))
print("\nMissing values:")
print(df_raw.isnull().sum())
print("\nSample document:")
if len(df_raw) > 0:
    sample = df_raw.iloc[0].to_dict()
    for k, v in sample.items():
        print(f"  {k}: {str(v)[:80]}")


## 4. Preprocessing

### 4.1 Cleaning Pipeline

Each document goes through three cleaning stages:
1. **HTML removal** — strip any leftover tags from scraping
2. **Symbol normalization** — remove ads, nav text, special characters
3. **Whitespace normalization** — collapse multiple spaces/newlines


In [ ]:
import re

# ── Cleaning functions ────────────────────────────────────────────────────────

HTML_TAG_RE    = re.compile(r"<[^>]+>")
MULTI_SPACE_RE = re.compile(r"\s{2,}")
URL_RE         = re.compile(r"https?://\S+")

# Noise phrases common in scraped web content
NOISE_PHRASES = [
    "subscribe to newsletter", "sign up for free", "advertisement",
    "cookie policy", "privacy policy", "all rights reserved",
    "click here", "read more", "share this", "follow us on",
    "loading...", "javascript is required", "enable cookies",
    "did you make this project", "related articles", "you might also like",
]

def clean_text(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""
    
    # 1. Remove HTML tags
    text = HTML_TAG_RE.sub(" ", text)
    
    # 2. Remove noise phrases (case-insensitive)
    for phrase in NOISE_PHRASES:
        text = re.sub(re.escape(phrase), " ", text, flags=re.IGNORECASE)
    
    # 3. Normalize unicode — keep standard ASCII + common unicode
    text = text.encode("utf-8", errors="ignore").decode("utf-8")
    
    # 4. Remove excessive symbols (keep letters, digits, basic punctuation)
    text = re.sub(r"[|]{2,}", " ", text)          # multiple pipes
    text = re.sub(r"[=]{3,}", " ", text)           # horizontal rules ===
    text = re.sub(r"[-]{3,}", " ", text)           # horizontal rules ---
    text = re.sub(r"[#]{2,}", " ", text)           # multiple hashes
    
    # 5. Normalize whitespace
    text = MULTI_SPACE_RE.sub(" ", text)
    text = text.strip()
    
    return text


# ── Apply cleaning ────────────────────────────────────────────────────────────

df_raw["content_raw_len"]   = df_raw["content"].apply(lambda x: len(str(x)) if x else 0)
df_raw["content_clean"]     = df_raw["content"].apply(clean_text)
df_raw["content_clean_len"] = df_raw["content_clean"].apply(len)

# Drop documents that are too short after cleaning
MIN_CONTENT_LEN = 80
before = len(df_raw)
df_clean = df_raw[df_raw["content_clean_len"] >= MIN_CONTENT_LEN].copy()
after = len(df_clean)

print(f"Documents before cleaning: {before}")
print(f"Documents after cleaning:  {after}")
print(f"Removed (too short):       {before - after}")
print(f"\nAvg content length (raw):   {df_raw['content_raw_len'].mean():.0f} chars")
print(f"Avg content length (clean): {df_clean['content_clean_len'].mean():.0f} chars")


### 4.2 Chunking

Long documents are split into ~500-token chunks (~2000 characters) with 50-token overlap.
This matches the context window requirements of our RAG pipeline.


In [ ]:
import uuid

CHUNK_SIZE = 2000
OVERLAP    = 200
MIN_CHUNK  = 100

SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")

def split_sentences(text):
    parts = SENTENCE_RE.split(text)
    return [p.strip() for p in parts if p.strip()]

def chunk_text(text):
    sentences = split_sentences(text)
    if not sentences:
        return []
    
    chunks, start = [], 0
    while start < len(sentences):
        current_len, end = 0, start
        while end < len(sentences):
            added = len(sentences[end]) + 1
            if current_len + added > CHUNK_SIZE and end > start:
                break
            current_len += added
            end += 1
        
        chunk_str = " ".join(sentences[start:end])
        if len(chunk_str) >= MIN_CHUNK:
            chunks.append(chunk_str)
        
        if end >= len(sentences):
            break
        
        # Overlap backtrack
        overlap_acc, new_start = 0, end
        while new_start > start + 1:
            new_start -= 1
            overlap_acc += len(sentences[new_start]) + 1
            if overlap_acc >= OVERLAP:
                break
        start = max(new_start, start + 1)
    
    return chunks


# ── Chunk all cleaned documents ───────────────────────────────────────────────

all_chunks = []
for _, row in df_clean.iterrows():
    raw_chunks = chunk_text(row["content_clean"])
    total = len(raw_chunks)
    
    for i, chunk_str in enumerate(raw_chunks):
        all_chunks.append({
            "chunk_id":     str(uuid.uuid4()),
            "source":       row.get("source", "unknown"),
            "domain":       row.get("domain", "general"),
            "roadmap":      row.get("roadmap", ""),
            "topic":        row.get("topic", ""),
            "url":          row.get("url", ""),
            "content_type": row.get("content_type", "text"),
            "difficulty":   row.get("difficulty", ""),
            "content":      chunk_str,
            "chunk_index":  i,
            "chunk_total":  total,
            "char_count":   len(chunk_str),
        })

df_chunks = pd.DataFrame(all_chunks)

print(f"Documents → Chunks")
print(f"  Input documents:  {len(df_clean)}")
print(f"  Output chunks:    {len(df_chunks)}")
print(f"  Avg chunks/doc:   {len(df_chunks)/len(df_clean):.1f}")
print(f"  Avg chunk length: {df_chunks['char_count'].mean():.0f} chars")
print(f"  Min chunk length: {df_chunks['char_count'].min()} chars")
print(f"  Max chunk length: {df_chunks['char_count'].max()} chars")


### 4.3 Metadata Labeling

Every chunk is tagged with:
- **source** — where it came from (roadmap.sh, freecodecamp, youtube, etc.)
- **domain** — learning domain (frontend, python, data_science, cooking, etc.)
- **content_type** — format (tutorial_article, video_transcript, course_syllabus, etc.)
- **difficulty** — beginner / intermediate / advanced (where available)


In [ ]:
# ── Difficulty inference for chunks without it ───────────────────────────────

BEGINNER_KEYWORDS    = ["beginner", "introduction", "basics", "getting started", "101", "fundamental", "overview", "learn to"]
INTERMEDIATE_KEYWORDS = ["intermediate", "advanced concepts", "deeper dive", "best practices", "patterns", "optimization"]
ADVANCED_KEYWORDS    = ["advanced", "expert", "system design", "architecture", "production", "at scale", "distributed"]

def infer_difficulty(row):
    if row.get("difficulty") and row["difficulty"] in ("beginner", "intermediate", "advanced"):
        return row["difficulty"]
    
    text = (str(row.get("topic", "")) + " " + str(row.get("content", ""))[:200]).lower()
    
    if any(k in text for k in ADVANCED_KEYWORDS):
        return "advanced"
    if any(k in text for k in INTERMEDIATE_KEYWORDS):
        return "intermediate"
    if any(k in text for k in BEGINNER_KEYWORDS):
        return "beginner"
    return "beginner"   # default

df_chunks["difficulty"] = df_chunks.apply(infer_difficulty, axis=1)

print("Difficulty distribution:")
print(df_chunks["difficulty"].value_counts().to_string())
print()
print("Sample labeled chunk:")
sample = df_chunks.iloc[0]
for col in ["chunk_id", "source", "domain", "topic", "content_type", "difficulty", "char_count"]:
    print(f"  {col:15}: {str(sample.get(col, ''))[:60]}")

## 5. ChromaDB — Embedding & Storage

Chunks are embedded using `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional vectors)
and stored in a persistent ChromaDB collection with cosine similarity.


In [ ]:
# ── ChromaDB storage demo ─────────────────────────────────────────────────────
# Full pipeline: backend/rag/embedder.py
# Here we show the connection and verify it works.

try:
    import chromadb
    from chromadb.utils import embedding_functions

    client = chromadb.PersistentClient(path="./chroma_db")
    
    try:
        ef = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        print("Embedding function: SentenceTransformer (all-MiniLM-L6-v2)")
    except Exception:
        ef = embedding_functions.DefaultEmbeddingFunction()
        print("Embedding function: ChromaDB default (ONNX MiniLM)")
    
    collection = client.get_or_create_collection(
        name="roadmapify_kb",
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    existing_count = collection.count()
    print(f"Collection: roadmapify_kb")
    print(f"Documents already stored: {existing_count}")
    
    # Store a small demo batch (first 20 chunks)
    demo_chunks = df_chunks.head(20).to_dict("records")
    
    collection.upsert(
        ids=[c["chunk_id"] for c in demo_chunks],
        documents=[c["content"] for c in demo_chunks],
        metadatas=[{k: v for k, v in c.items() if k not in ("chunk_id", "content")} for c in demo_chunks],
    )
    
    new_count = collection.count()
    print(f"\nAfter demo upsert: {new_count} documents")
    
    # Test retrieval
    results = collection.query(
        query_texts=["how to learn React for beginners"],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )
    
    print("\nTest query: 'how to learn React for beginners'")
    print("Top 3 results:")
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"  [{meta.get('domain'):15}] {meta.get('topic','')[:40]:40} (dist={dist:.4f})")

except Exception as e:
    print(f"ChromaDB demo: {e}")
    print("Run backend/rag/embedder.py to populate the full collection.")


## 6. Exploratory Data Analysis (EDA)

### 6.1 Document Distribution by Source


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Chunks by source ────────────────────────────────────────────────────
source_counts = df_chunks["source"].value_counts()
colors = PALETTE[:len(source_counts)]

axes[0].barh(source_counts.index, source_counts.values, color=colors)
axes[0].set_xlabel("Number of Chunks")
axes[0].set_title("Chunks by Source", fontsize=13, fontweight="bold", pad=12)
for i, v in enumerate(source_counts.values):
    axes[0].text(v + 2, i, str(v), va="center", fontsize=9)

# ── Right: Chunks by content type ────────────────────────────────────────────
ct_counts = df_chunks["content_type"].value_counts()
wedge_props = dict(width=0.5, edgecolor="white", linewidth=2)
axes[1].pie(
    ct_counts.values,
    labels=ct_counts.index,
    autopct="%1.1f%%",
    colors=PALETTE[:len(ct_counts)],
    wedgeprops=wedge_props,
    startangle=90,
)
axes[1].set_title("Chunks by Content Type", fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig("eda_source_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nTotal sources represented: {df_chunks['source'].nunique()}")
print(f"Total content types:       {df_chunks['content_type'].nunique()}")


### 6.2 Domain Coverage

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

domain_counts = df_chunks["domain"].value_counts()
colors = [PALETTE[i % len(PALETTE)] for i in range(len(domain_counts))]

bars = ax.bar(range(len(domain_counts)), domain_counts.values, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xticks(range(len(domain_counts)))
ax.set_xticklabels(domain_counts.index, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Number of Chunks")
ax.set_title("Knowledge Base Coverage by Domain", fontsize=14, fontweight="bold", pad=12)

for bar, val in zip(bars, domain_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("eda_domain_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Domains covered: {df_chunks['domain'].nunique()}")
print(f"\nTop 5 domains:")
print(domain_counts.head().to_string())


### 6.3 Chunk Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Histogram ────────────────────────────────────────────────────────────────
axes[0].hist(df_chunks["char_count"], bins=40, color=PALETTE[0], edgecolor="white", linewidth=0.5)
axes[0].axvline(df_chunks["char_count"].mean(), color=PALETTE[1], linestyle="--", linewidth=1.5, label=f'Mean: {df_chunks["char_count"].mean():.0f}')
axes[0].axvline(df_chunks["char_count"].median(), color=PALETTE[2], linestyle="--", linewidth=1.5, label=f'Median: {df_chunks["char_count"].median():.0f}')
axes[0].set_xlabel("Characters per Chunk")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Chunk Length Distribution", fontsize=13, fontweight="bold")
axes[0].legend()

# ── Box plot by difficulty ────────────────────────────────────────────────────
diff_order = ["beginner", "intermediate", "advanced"]
diff_data  = [df_chunks[df_chunks["difficulty"] == d]["char_count"].values for d in diff_order]
bp = axes[1].boxplot(diff_data, labels=diff_order, patch_artist=True, medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel("Characters per Chunk")
axes[1].set_title("Chunk Length by Difficulty", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("eda_chunk_lengths.png", dpi=150, bbox_inches="tight")
plt.show()

print("Chunk length statistics:")
print(df_chunks["char_count"].describe().round(1).to_string())


### 6.4 Difficulty Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Overall difficulty ────────────────────────────────────────────────────────
diff_counts = df_chunks["difficulty"].value_counts()
diff_colors = {"beginner": PALETTE[2], "intermediate": PALETTE[0], "advanced": PALETTE[1]}
bar_colors  = [diff_colors.get(d, PALETTE[3]) for d in diff_counts.index]

axes[0].bar(diff_counts.index, diff_counts.values, color=bar_colors, edgecolor="white", linewidth=0.5)
axes[0].set_ylabel("Number of Chunks")
axes[0].set_title("Chunks by Difficulty Level", fontsize=13, fontweight="bold")
for i, v in enumerate(diff_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontsize=10)

# ── Difficulty breakdown by domain (top 8 domains) ───────────────────────────
top_domains = df_chunks["domain"].value_counts().head(8).index
df_top = df_chunks[df_chunks["domain"].isin(top_domains)]
pivot = df_top.groupby(["domain", "difficulty"]).size().unstack(fill_value=0)

# Ensure all difficulty columns exist
for col in ["beginner", "intermediate", "advanced"]:
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[["beginner", "intermediate", "advanced"]]

pivot.plot(
    kind="bar", ax=axes[1], stacked=True,
    color=[diff_colors["beginner"], diff_colors["intermediate"], diff_colors["advanced"]],
    edgecolor="white", linewidth=0.3,
)
axes[1].set_xlabel("")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=35, ha="right", fontsize=8)
axes[1].set_ylabel("Number of Chunks")
axes[1].set_title("Difficulty by Domain (Top 8)", fontsize=13, fontweight="bold")
axes[1].legend(title="Difficulty", loc="upper right")

plt.tight_layout()
plt.savefig("eda_difficulty.png", dpi=150, bbox_inches="tight")
plt.show()


### 6.5 Most Frequent Terms

In [ ]:
from collections import Counter
import re

STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "is", "are", "was", "be", "have", "has", "do", "does",
    "it", "this", "that", "you", "we", "i", "can", "will", "how", "what",
    "your", "our", "from", "by", "as", "if", "so", "not", "all", "more",
    "also", "one", "two", "use", "used", "using", "new", "get", "its",
    "their", "they", "which", "when", "where", "would", "make", "like",
    "into", "than", "then", "just", "about", "up", "out", "some", "each",
    "other", "these", "those", "after", "before", "through", "any", "no",
}

def get_top_terms(series, n=20):
    words = []
    for text in series.dropna():
        tokens = re.findall(r"\b[a-z]{4,}\b", str(text).lower())
        words.extend(t for t in tokens if t not in STOP_WORDS)
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Overall top terms ─────────────────────────────────────────────────────────
top_terms = get_top_terms(df_chunks["content"])
terms, counts = zip(*top_terms)
axes[0].barh(list(reversed(terms)), list(reversed(counts)), color=PALETTE[0])
axes[0].set_title("Top 20 Terms (All Domains)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Frequency")

# ── Top terms in tech vs non-tech ────────────────────────────────────────────
TECH_DOMAINS    = ["frontend","backend","python","javascript","data_science","devops","sql","dsa","git","computer_science","full-stack"]
NON_TECH_DOMAINS = ["cooking","baking","crochet","ielts_preparation","language_learning","digital_marketing","uiux_design"]

df_tech    = df_chunks[df_chunks["domain"].isin(TECH_DOMAINS)]
df_nontech = df_chunks[df_chunks["domain"].isin(NON_TECH_DOMAINS)]

tech_terms    = get_top_terms(df_tech["content"],    n=10) if len(df_tech)    > 0 else []
nontech_terms = get_top_terms(df_nontech["content"], n=10) if len(df_nontech) > 0 else []

y = range(len(tech_terms))
if tech_terms:
    terms_t, counts_t = zip(*tech_terms)
    axes[1].barh([i + 0.2 for i in y], counts_t, height=0.4, color=PALETTE[0], label="Tech domains")
if nontech_terms:
    terms_n, counts_n = zip(*nontech_terms)
    axes[1].barh([i - 0.2 for i in range(len(nontech_terms))], counts_n, height=0.4, color=PALETTE[2], label="Non-tech domains")
    axes[1].set_yticks(range(max(len(tech_terms), len(nontech_terms))))
    axes[1].set_yticklabels(list(terms_t)[:max(len(tech_terms), len(nontech_terms))], fontsize=8)

axes[1].set_title("Top Terms: Tech vs Non-Tech", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_top_terms.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Final Dataset Summary

In [ ]:
print("=" * 65)
print("  ROADMAPIFY — PHASE 2 DATASET SUMMARY")
print("=" * 65)

print(f"\n  Raw documents collected:     {len(all_raw_docs):>6}")
print(f"  After cleaning:              {len(df_clean):>6}")
print(f"  Total chunks (indexed):      {len(df_chunks):>6}")
print(f"  Unique domains covered:      {df_chunks['domain'].nunique():>6}")
print(f"  Unique sources:              {df_chunks['source'].nunique():>6}")
print(f"  Avg chunk size:              {df_chunks['char_count'].mean():>6.0f} chars")
print(f"  Avg chunks per document:     {len(df_chunks)/max(len(df_clean),1):>6.1f}")

print("\n  Difficulty breakdown:")
for diff, count in df_chunks["difficulty"].value_counts().items():
    pct = count / len(df_chunks) * 100
    bar = "█" * int(pct / 3)
    print(f"    {diff:>12}: {count:>5}  ({pct:.1f}%)  {bar}")

print("\n  Domain breakdown:")
for domain, count in df_chunks["domain"].value_counts().head(10).items():
    print(f"    {domain:>25}: {count:>5} chunks")

print("\n  Sources in knowledge base:")
for src, count in df_chunks["source"].value_counts().items():
    print(f"    {src:>25}: {count:>5} chunks")

print("\n  Ready for embedding into ChromaDB.")
print("  Run: python backend/rag/embedder.py")
print("=" * 65)


## 8. Save Processed Chunks

In [ ]:
# Save the final chunked + labeled dataset
output_path = pathlib.Path("data/processed/all_chunks.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

chunks_to_save = df_chunks.to_dict("records")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks_to_save, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(chunks_to_save)} chunks → {output_path}")
print("\nPhase 2 complete. Files produced:")
print("  data/processed/all_chunks.json  — full chunk dataset")
print("  chroma_db/                       — ChromaDB persistent store (demo subset)")
print("  eda_source_distribution.png")
print("  eda_domain_distribution.png")
print("  eda_chunk_lengths.png")
print("  eda_difficulty.png")
print("  eda_top_terms.png")
